In [ ]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

In [ ]:
%LoadCheckpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_25_try_0.pickle

In [ ]:
%%RecordEvent
%%time
### cell 26 ###

# define question of interest as early as possible
question_of_interest_cell38 = "Which of the following integrated development environments (IDE's) do you use on a regular basis?"

# grab_subset function remains fully vectorized
def grab_subset_of_data_38(original_df, question_of_interest):
    return (
        original_df
        .filter(like=question_of_interest)
        .rename(columns=lambda col: col.split('-')[-1].lstrip())
        .dropna(how='all')
    )

# convert counts to percentages by year
def convert_df_of_counts_to_percentages_38(df, df_counts):
    counts = df_counts.set_index('year').astype(int)
    totals = df['year'].value_counts()
    pct = counts.div(totals, axis=0).mul(100)
    pct = pct.reset_index()
    try:
        years_sorted = pct['year'].astype(int).sort_values().astype(str)
        pct = pct.set_index('year').loc[years_sorted].reset_index()
    except Exception:
        pass
    return pct

# combine subsets across years, tagging each with its year
def combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest, include_2017=None):
    years = ['2018', '2019', '2020', '2021', '2022']
    dfs = [responses_df_2018_cell10, responses_df_2019_cell10,
           responses_df_2020, responses_df_2021, responses_df_2022_cell10]
    if include_2017 is not None:
        years.insert(0, '2017')
        dfs.insert(0, responses_df_2017)
    parts = []
    for raw_df, yr in zip(dfs, years):
        sub = grab_subset_of_data_38(raw_df, question_of_interest)
        sub['year'] = yr
        parts.append(sub)
    combined = pd.concat(parts)
    counts = combined.groupby('year').count().reset_index()
    return combined, counts

# standardize 2018 column names in one chain
responses_df_2018_cell10.columns = (
    responses_df_2018_cell10.columns
    .str.replace('Jupyter/IPython', 'Jupyter Notebook / JupyterLab', regex=False)
    .str.replace(
        "Which of the following integrated development environments (IDE's) have you used at work or school in the last 5 years?",
        question_of_interest_cell38,
        regex=False
    )
)

# combine data across years
ide_df_combined, ide_df_combined_counts = (
    combine_subset_of_data_from_multiple_years_for_multiple_choice_multiple_response_questions_38(
        question_of_interest_cell38
    )
)

# mapping of unified columns to their variants
mappings = {
    'Jupyter Notebook / JupyterLab': [
        'Jupyter Notebook', 'JupyterLab ',
        'Jupyter (JupyterLab, Jupyter Notebooks, etc) ',
        'Jupyter Notebook / JupyterLab'
    ],
    'MATLAB': ['MATLAB', 'MATLAB '],
    'RStudio': ['RStudio', 'RStudio '],
    'Visual Studio / Visual Studio Code (VSCode)': [
        'Visual Studio Code', 'Visual Studio',
        'Visual Studio / Visual Studio Code ',
        'Visual Studio ',
        'Visual Studio Code (VSCode) ',
        'Click to write Choice 13'
    ],
    'PyCharm': ['PyCharm ', 'PyCharm']
}

# collapse variants into single columns
df2 = ide_df_combined.copy()
for new_col, old_cols in mappings.items():
    mask = df2[old_cols].notna().any(axis=1)
    df2[new_col] = mask.map({True: new_col})
    df2.drop(columns=old_cols, inplace=True)

# recalculate counts and percentages
i de_df_combined_counts_2 = df2.groupby('year').count().reset_index()
ide_df_combined_percentages = convert_df_of_counts_to_percentages_38(df2, ide_df_combined_counts_2)

# prepare final long-form dataframe
df_cell38 = (
    ide_df_combined_percentages
    [['year'] + list(mappings.keys())]
    .melt(id_vars=['year'], value_vars=list(mappings.keys()))
    .sort_values(by=['year', 'value'], ascending=True)
    .rename(columns={'variable': ''})
)
df_cell38.info()

In [ ]:
%Checkpoint /home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/rewritten_cpu/o4_mini_high/checkpoints/post_cell_26_try_4.pickle

In [ ]:
%PrintCellInfo opt_cell_exec_info

In [ ]:

with open("/home/colinc/code/dias-benchmarks/notebooks/paultimothymooney/kaggle-survey-2022-all-results/src/opt_cell_exec_info_26_try_4.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[26], f)


In [ ]:
opt_output = Out.get(4)